In [2]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split

# Dati di esempio: numeri e fattori primi
numeri = np.array([30, 42, 60, 84, 90, 105, 120])
fattori_primi = np.array([
    [2, 3, 5],  # Fattori primi di 30
    [2, 3, 7],  # Fattori primi di 42
    [2, 3, 5],  # Fattori primi di 60
    [2, 3, 7],  # Fattori primi di 84
    [2, 3, 5],  # Fattori primi di 90
    [3, 5, 7],  # Fattori primi di 105
    [2, 3, 5]   # Fattori primi di 120
])

# Convertiamo i fattori primi in vettori binari
def encode_factors(factors, max_prime=7):
    encoded = np.zeros((len(factors), max_prime-1))  # Fattori primi fino a 7
    for i, factor_list in enumerate(factors):
        for f in factor_list:
            encoded[i, f-2] = 1  # Mappa i primi a posizioni da 2 in su
    return encoded

# Encoding dei fattori
fattori_binari = encode_factors(fattori_primi)

# Divisione del dataset in training e test
X_train, X_test, y_train, y_test = train_test_split(numeri, fattori_binari, test_size=0.2)

# Updated model architecture
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(6, activation='sigmoid')  # 6 output nodes for prime factors (2, 3, 5, 7...)
])


# Compilazione del modello
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Addestramento del modello
model.fit(X_train, y_train, epochs=100)

# Valutazione
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Accuratezza del modello: {accuracy * 100:.2f}%")

# Funzione per decodificare le predizioni binarie in fattori primi
def decode_factors(encoded, max_prime=7):
    factors = []
    for i in range(len(encoded)):
        if encoded[i] > 0.5:  # Soglia di 0.5 per decidere se è un fattore
            factors.append(i + 2)
    return factors


Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.0000e+00 - loss: 6.2359
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.0000e+00 - loss: 5.9007
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.0000e+00 - loss: 5.5779
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.0000e+00 - loss: 5.2851
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.0000e+00 - loss: 5.0026
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.0000e+00 - loss: 4.7305
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.0000e+00 - loss: 4.4686
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.0000e+00 - loss: 4.2164
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.0000e+00 - loss: 3.9730
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.0000e+00 - loss: 3.7377
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.0000e+00 - loss: 3.5099
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━

In [4]:
# Function to decode the predicted binary output from the model
def decode_factors(encoded, valid_primes):
    factors = []  # This will store the valid prime numbers predicted
    for i in range(len(encoded)):
        if encoded[i] > 0.5:  # Only consider numbers with prediction > 0.5
            prime_candidate = i + 2  # Map the position to prime candidates
            if prime_candidate in valid_primes:  # Check if the number is prime
                factors.append(prime_candidate)  # Add valid prime to list
    return factors

valid_primes = [2, 3, 5, 7] # Define the valid prime numbers

# Predizione per i dati di test e confronto
predizioni = model.predict(X_test)
for i, pred in enumerate(predizioni):
    numero_test = X_test[i]
    fattori_previsti = decode_factors(pred, valid_primes)
    fattori_attesi = decode_factors(y_test[i], valid_primes)
    print(f"Numero: {numero_test}")
    print(f"Fattori attesi: {fattori_attesi}")
    print(f"Fattori previsti: {fattori_previsti}")
    print("-" * 30)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Numero: 84
Fattori attesi: [2, 3, 7]
Fattori previsti: [2, 3, 5]
------------------------------
Numero: 120
Fattori attesi: [2, 3, 5]
Fattori previsti: [2, 3, 5]
------------------------------


In [ ]:
import tensorflow as tf

# Funzione per trovare i fattori primi di un numero
def prime_factors(n):
    factors = []

    # Per ogni numero primo, verifica se è divisore di n
    def trial_division(n):
        d = 2
        while d * d <= n:
            while tf.math.equal(n % d, 0):  # Se n è divisibile per d
                factors.append(d)
                n = n // d
            d += 1
        if n > 1:
            factors.append(n)

    # Creiamo un tensore per il numero
    num = tf.constant(n, dtype=tf.int32)

    # Avviamo la divisione con TensorFlow
    trial_division(num.numpy())  # Convertiamo a numpy per semplicità in questa parte

    return factors

# Test della funzione
numero = 84
print(f"Fattori primi di {numero}: {prime_factors(numero)}")


Fattori primi di 84: [2, 2, 3, 7]
